# Nemotron-3-Nano-30B SFT Training
Fine-tune with LoRA on Chain-of-Thought reasoning traces for logical puzzle solving.

In [ ]:
import json
import os
import sys
import site
import subprocess
import glob
from pathlib import Path

# Fix cutlass/mamba_ssm import issue
# The nvidia-utility-script provides mamba_ssm and cutlass packages
# We need to add the correct paths BEFORE importing anything

# Find and add all potential cutlass/nvidia package paths
utility_base = "/kaggle/usr/lib/notebooks/ryanholbrook/nvidia-utility-script"
utility_base_alt = "/kaggle/usr/lib/notebooks/ryanholbrook/nvidia_utility_script"

for base in [utility_base, utility_base_alt]:
    if os.path.exists(base):
        print(f"Found utility script at: {base}")
        # Add the base dir itself
        site.addsitedir(base)
        # Add the cutlass python packages subdir
        cutlass_path = os.path.join(base, "nvidia_cutlass_dsl", "python_packages")
        if os.path.exists(cutlass_path):
            site.addsitedir(cutlass_path)
            print(f"Added cutlass path: {cutlass_path}")
        # Add any other python_packages dirs
        for pp in glob.glob(os.path.join(base, "**/python_packages"), recursive=True):
            if pp not in sys.path:
                site.addsitedir(pp)
                print(f"Added path: {pp}")

# Verify cutlass is importable
try:
    import cutlass
    print(f"cutlass imported OK from: {cutlass.__file__}")
except ImportError as e:
    print(f"WARNING: cutlass not importable: {e}")
    print("Searching for cutlass...")
    for root, dirs, files in os.walk("/kaggle/usr/lib"):
        if "cutlass" in dirs or any(f.startswith("cutlass") for f in files):
            print(f"  Found: {root}")

OUTPUT_DIR = Path("/kaggle/working")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Hyperparameters
LORA_RANK = 32
LORA_ALPHA = 64
LORA_DROPOUT = 0.05
TARGET_MODULES = r".*\.(in_proj|out_proj|up_proj|down_proj)$"
LEARNING_RATE = 2e-5
NUM_EPOCHS = 3
BATCH_SIZE = 1
GRADIENT_ACCUMULATION_STEPS = 8
MAX_SEQ_LENGTH = 2048
WARMUP_RATIO = 0.05

print(f"\nConfig: LoRA r={LORA_RANK} a={LORA_ALPHA}, LR={LEARNING_RATE}, epochs={NUM_EPOCHS}")
print(f"Effective batch: {BATCH_SIZE * GRADIENT_ACCUMULATION_STEPS}")

In [ ]:
# Load model
import kagglehub
import mamba_ssm
import torch
from peft import LoraConfig, get_peft_model, TaskType
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_PATH = kagglehub.model_download(
    "metric/nemotron-3-nano-30b-a3b-bf16/transformers/default"
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_PATH,
    device_map="auto",
    trust_remote_code=True,
    torch_dtype=torch.bfloat16,
)
tokenizer = AutoTokenizer.from_pretrained(
    MODEL_PATH,
    trust_remote_code=True,
)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
    tokenizer.pad_token_id = tokenizer.eos_token_id

print(f"Model loaded. Vocab size: {len(tokenizer)}")
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")

# Apply LoRA
lora_config = LoraConfig(
    r=LORA_RANK,
    lora_alpha=LORA_ALPHA,
    target_modules=TARGET_MODULES,
    lora_dropout=LORA_DROPOUT,
    bias="none",
    task_type=TaskType.CAUSAL_LM,
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

In [ ]:
# Load CoT training data - search for the file in common Kaggle paths
import glob as _glob

COT_CANDIDATES = [
    "/kaggle/input/nemotron-cot-training-data/cot_train.jsonl",
    "/kaggle/input/nemotron-cot-training-data/nemotron-cot-data/cot_train.jsonl",
]
# Also search recursively
for f in _glob.glob("/kaggle/input/**/cot_train.jsonl", recursive=True):
    if f not in COT_CANDIDATES:
        COT_CANDIDATES.append(f)

COT_PATH = None
for p in COT_CANDIDATES:
    if os.path.exists(p):
        COT_PATH = Path(p)
        break

if COT_PATH is None:
    # List what's actually in /kaggle/input to debug
    print("ERROR: cot_train.jsonl not found!")
    print("Available input dirs:")
    for d in sorted(Path("/kaggle/input").iterdir()):
        print(f"  {d.name}/")
        for f in sorted(d.iterdir())[:10]:
            print(f"    {f.name} ({f.stat().st_size / 1024:.1f} KB)")
    raise FileNotFoundError("cot_train.jsonl not found in any expected path")

print(f"Found CoT data at: {COT_PATH}")

examples = []
with open(COT_PATH) as f:
    for line in f:
        if line.strip():
            examples.append(json.loads(line))

print(f"Loaded {len(examples)} CoT examples")

# Stats
from collections import Counter
type_counts = Counter(ex['type'] for ex in examples)
for t, c in type_counts.most_common():
    print(f"  {t}: {c}")

# Format as chat
texts = []
for ex in examples:
    text = tokenizer.apply_chat_template(
        ex["messages"],
        tokenize=False,
        add_generation_prompt=False,
    )
    texts.append(text)

# Token length analysis
sample_lengths = [len(tokenizer.encode(t)) for t in texts[:200]]
import statistics
print(f"\nToken lengths (sample): mean={statistics.mean(sample_lengths):.0f}, "
      f"max={max(sample_lengths)}, p95={sorted(sample_lengths)[int(len(sample_lengths)*0.95)]}")

In [ ]:
# Create dataset
from torch.utils.data import Dataset, random_split

class TextDataset(Dataset):
    def __init__(self, texts, tokenizer, max_length):
        self.encodings = []
        skipped = 0
        for text in texts:
            enc = tokenizer(
                text,
                truncation=True,
                max_length=max_length,
                padding="max_length",
                return_tensors="pt",
            )
            actual_len = (enc["input_ids"] != tokenizer.pad_token_id).sum().item()
            if actual_len < max_length * 0.98:
                self.encodings.append({k: v.squeeze(0) for k, v in enc.items()})
            else:
                skipped += 1
        print(f"Dataset: {len(self.encodings)} examples ({skipped} skipped)")

    def __len__(self):
        return len(self.encodings)

    def __getitem__(self, idx):
        item = {k: v.clone() for k, v in self.encodings[idx].items()}
        item["labels"] = item["input_ids"].clone()
        item["labels"][item["labels"] == tokenizer.pad_token_id] = -100
        return item

dataset = TextDataset(texts, tokenizer, MAX_SEQ_LENGTH)

val_size = max(1, len(dataset) // 20)
train_size = len(dataset) - val_size
train_dataset, val_dataset = random_split(
    dataset, [train_size, val_size],
    generator=torch.Generator().manual_seed(42),
)
print(f"Train: {len(train_dataset)}, Val: {len(val_dataset)}")

In [ ]:
# Training
from transformers import TrainingArguments, Trainer

training_args = TrainingArguments(
    output_dir=str(OUTPUT_DIR / "checkpoints"),
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
    learning_rate=LEARNING_RATE,
    lr_scheduler_type="cosine",
    warmup_ratio=WARMUP_RATIO,
    weight_decay=0.01,
    bf16=True,
    logging_steps=10,
    eval_strategy="steps",
    eval_steps=100,
    save_strategy="steps",
    save_steps=200,
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    gradient_checkpointing=True,
    optim="adamw_torch_fused",
    dataloader_pin_memory=True,
    report_to="none",
    max_grad_norm=1.0,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
)

print("Starting training...")
trainer.train()
print("Training complete!")

In [ ]:
# Save adapter and create submission.zip
print("Saving LoRA adapter...")
model.save_pretrained(str(OUTPUT_DIR))

adapter_config = OUTPUT_DIR / "adapter_config.json"
adapter_model = OUTPUT_DIR / "adapter_model.safetensors"
assert adapter_config.exists(), f"Missing {adapter_config}"
assert adapter_model.exists(), f"Missing {adapter_model}"

print(f"adapter_config.json: {adapter_config.stat().st_size / 1024:.1f} KB")
print(f"adapter_model.safetensors: {adapter_model.stat().st_size / 1024 / 1024:.1f} MB")

os.chdir(str(OUTPUT_DIR))
subprocess.run(
    ["zip", "submission.zip", "adapter_config.json", "adapter_model.safetensors"],
    check=True,
)
print(f"\nsubmission.zip: {os.path.getsize('submission.zip') / 1024 / 1024:.1f} MB")
print("Done!")